In [24]:
###############################################################################
# DRL for Quantum Circuit Shot Allocation
# ----------------------------------------
# Aligned with the IncExc_to_TQC paper (Bisicchia et al., 2026):
#   • TEST SET = exact 36 traces from Tables 8-13 of the paper
#     (each algorithm uses specific size/backend pairs, NOT a full cartesian product)
#   • TRAIN SET = 864 other traces from qsimbench — zero data leakage
#   • AgentType controls training AND mid-training validation distribution:
#       EASY       → 100% easy training & validation
#       HARD       → 100% hard training & validation
#       GENERIC    → 50/50 balanced mix (train & validation)
#       UNBALANCED → 20% easy / 80% hard (train & validation)
#   • Final evaluation ALWAYS uses ALL available problems (50% easy, 50% hard):
#       - Eval set: all 36 paper problems (18 easy + 18 hard)
#       - Train-set eval: all 864 training problems (overfitting diagnostic)
#   • Small/Large classification from Ch. 6.1:
#       SplitMetric.SIZE   → size < 10 → easy, size >= 10 → hard
#       SplitMetric.ORACLE → oracle <= 5000 → easy, oracle > 5000 → hard
#   • Best-model checkpointing (every 100 eps from ep 2000, keep lowest val MAE)
#   • Early stopping with patience parameter
#   • MAE evaluated on the held-out 36 paper problems
#   • Error percentage = MAE / max_shots
#   • SVG outputs with easy/hard tagging
#   • Oracle = a posteriori optimal (paper definition, §6.1):
#       "minimum n such that D(P̂_n, P̂_B) ≤ δ"  with B=20000, δ=0.05
###############################################################################

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque, Counter
import random
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gym
from gym import spaces
from qsimbench import get_outcomes, get_index
from typing import Dict, List, Tuple, Optional
from enum import Enum
from tqdm import tqdm
import os
import json
from ast import literal_eval
import copy
from functools import lru_cache

# ─────────────────────────────────────────────────────────────────────────────
# Agent type: controls training distribution bias
# ─────────────────────────────────────────────────────────────────────────────

class AgentType(Enum):
    EASY       = "easy"        # trains ONLY on easy problems (size < 10)
    HARD       = "hard"        # trains ONLY on hard problems (size >= 10)
    GENERIC    = "generic"     # trains on a balanced 50/50 mix of easy and hard
    UNBALANCED = "unbalanced"  # trains on 20% easy, 80% hard

class RewardType(Enum):
    MAE        = "mae"         # exp(-0.003 * mae) for overshoot, -1 - mae/optimal for undershoot
    ASYMMETRIC = "asymmetric"  # exp(-0.00025 * error) for overshoot, (-1.85 - |e|/opt) * scale(1-1.5) for undershoot

class SplitMetric(Enum):
    SIZE   = "size"    # easy/hard split by circuit size (size < 10)
    ORACLE = "oracle"  # easy/hard split by oracle optimal shots (oracle <= 5000)

# ─────────────────────────────────────────────────────────────────────────────
# Paper-aligned constants (Ch. 6.1 of IncExc_to_TQC)
# ─────────────────────────────────────────────────────────────────────────────
# Exact 36 traces from Tables 8-13 (each algorithm uses specific size/backend pairs)
PAPER_TRACES: List[Tuple[str, int, str]] = [
    ("dj", 4, "fake_fez"), ("dj", 4, "fake_torino"),
    ("dj", 8, "fake_fez"), ("dj", 8, "fake_marrakesh"),
    ("dj", 10, "fake_fez"), ("dj", 10, "fake_torino"),
    ("dj", 14, "fake_fez"), ("dj", 14, "fake_marrakesh"),
    ("qaoa", 4, "fake_sherbrooke"),
    ("qaoa", 6, "fake_marrakesh"),
    ("qaoa", 8, "fake_kyiv"), ("qaoa", 8, "fake_sherbrooke"),
    ("qaoa", 10, "fake_sherbrooke"),
    ("qaoa", 12, "fake_marrakesh"),
    ("qaoa", 14, "fake_kyiv"), ("qaoa", 14, "fake_sherbrooke"),
    ("qft", 6, "fake_fez"), ("qft", 6, "fake_torino"),
    ("qft", 8, "fake_fez"), ("qft", 8, "fake_torino"),
    ("qft", 12, "fake_fez"), ("qft", 12, "fake_torino"),
    ("qft", 14, "fake_fez"), ("qft", 14, "fake_torino"),
    ("qnn", 8, "fake_fez"), ("qnn", 8, "fake_kyiv"),
    ("qnn", 14, "fake_fez"), ("qnn", 14, "fake_kyiv"),
    ("random", 6, "fake_fez"),
    ("random", 8, "fake_fez"),
    ("random", 12, "fake_fez"),
    ("random", 14, "fake_fez"),
    ("vqe", 6, "fake_kyiv"),
    ("vqe", 8, "fake_kyiv"),
    ("vqe", 12, "fake_kyiv"),
    ("vqe", 14, "fake_kyiv"),
]

SMALL_LARGE_THRESHOLD = 10   # size < 10 → easy, size >= 10 → hard
ORACLE_EASY_THRESHOLD = 5_000  # oracle <= 5000 → easy (when using SplitMetric.ORACLE)
MAX_SHOTS = 20_000
LOW_SHOT_THRESHOLD = 5_000   # counter: problems solved with < 5000 shots

# Active split metric — set in __main__ config block
SPLIT_METRIC = SplitMetric.SIZE

# ─────────────────────────────────────────────────────────────────────────────
# Cached qsimbench index  — called once, reused everywhere
# ─────────────────────────────────────────────────────────────────────────────
_INDEX_CACHE: Optional[dict] = None

def get_cached_index() -> dict:
    """Return qsimbench index, fetching it only once."""
    global _INDEX_CACHE
    if _INDEX_CACHE is None:
        _INDEX_CACHE = get_index()
    return _INDEX_CACHE

# ─────────────────────────────────────────────────────────────────────────────
# Global feature-encoding maps  — built ONCE from the full qsimbench index
# so that every environment (train / validation / eval) encodes the same
# algorithm / backend with the same normalised value.
# ─────────────────────────────────────────────────────────────────────────────
_GLOBAL_ALG_MAP: Optional[Dict[str, int]] = None
_GLOBAL_BACKEND_MAP: Optional[Dict[str, int]] = None

def get_global_mappings() -> Tuple[Dict[str, int], Dict[str, int]]:
    """Return (alg_map, backend_map) derived from the FULL qsimbench index.
    Cached after the first call."""
    global _GLOBAL_ALG_MAP, _GLOBAL_BACKEND_MAP
    if _GLOBAL_ALG_MAP is not None:
        return _GLOBAL_ALG_MAP, _GLOBAL_BACKEND_MAP
    index = get_cached_index()
    all_algs: set = set()
    all_backends: set = set()
    for alg in index:
        all_algs.add(alg)
        for sz in index[alg]:
            val = index[alg][sz]
            if isinstance(val, dict):
                all_backends.update(val.keys())
            else:
                all_backends.update(val)
    _GLOBAL_ALG_MAP     = {n: i for i, n in enumerate(sorted(all_algs))}
    _GLOBAL_BACKEND_MAP = {n: i for i, n in enumerate(sorted(all_backends))}
    return _GLOBAL_ALG_MAP, _GLOBAL_BACKEND_MAP

# ─────────────────────────────────────────────────────────────────────────────
# Oracle and caching system
# ─────────────────────────────────────────────────────────────────────────────
ORACLE_CACHE: Dict[Tuple[str, int, str], int] = {}
CACHE_FILE = "oracle_cache_enhanced.json"

def save_oracle_cache():
    """Persist the dictionary of computed optimal shots to JSON."""
    string_key_cache = {str(k): v for k, v in ORACLE_CACHE.items()}
    with open(CACHE_FILE, "w") as f:
        json.dump(string_key_cache, f)

def load_oracle_cache():
    """Load pre-computed optimal shots from the cache file."""
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, "r") as f:
                string_key_cache = json.load(f)
                for k_str, v in string_key_cache.items():
                    ORACLE_CACHE[literal_eval(k_str)] = v
            print(f"Loaded {len(ORACLE_CACHE)} items from oracle cache.")
        except Exception as e:
            print(f"Warning: Could not read cache file. Error: {e}")

def find_optimal_shots(
    algorithm: str, size: int, backend: str,
    step_size: int = 50, max_shots: int = MAX_SHOTS,
    delta: float = 0.05,
) -> int:
    """
    Oracle: a posteriori optimal shots (paper definition §6.1).

    Collects the FULL budget of `max_shots` shots first, producing the
    reference distribution P̂_B.  Then finds the smallest n (in steps of
    `step_size`) such that  TVD(P̂_n, P̂_B) ≤ δ.

    δ = 0.05 corresponds to the strictest column in the paper tables.
    """
    cache_key = (algorithm, size, backend)
    if cache_key in ORACLE_CACHE:
        return ORACLE_CACHE[cache_key]

    def normalize(counts: Dict[str, int]) -> Dict[str, float]:
        total = sum(counts.values())
        return {k: v / total for k, v in counts.items()} if total > 0 else {}

    def tvd(p: Dict[str, float], q: Dict[str, float]) -> float:
        all_keys = set(p) | set(q)
        return 0.5 * sum(abs(p.get(k, 0) - q.get(k, 0)) for k in all_keys)

    # 1. Collect ALL shots up to the full budget
    n_batches = max_shots // step_size
    batches: List[Dict[str, int]] = []
    for _ in range(n_batches):
        try:
            batch = get_outcomes(algorithm, size, backend,
                                 shots=step_size, strategy="random", exact=True)
            batches.append(batch)
        except Exception:
            ORACLE_CACHE[cache_key] = max_shots
            return max_shots

    # 2. Build FULL-BUDGET reference distribution P̂_B
    full_counts: Counter = Counter()
    for b in batches:
        full_counts.update(b)
    ref_dist = normalize(full_counts)

    # 3. Walk forward and find first n where TVD(P̂_n, P̂_B) ≤ δ
    cumulative: Counter = Counter()
    for i, b in enumerate(batches):
        cumulative.update(b)
        n = (i + 1) * step_size
        current_dist = normalize(cumulative)
        if tvd(current_dist, ref_dist) <= delta:
            ORACLE_CACHE[cache_key] = n
            return n

    ORACLE_CACHE[cache_key] = max_shots
    return max_shots

# ─────────────────────────────────────────────────────────────────────────────
# Problem sets: paper test set (36) vs training set (864)  — cached
# ─────────────────────────────────────────────────────────────────────────────

def classify_problem(triplet: Tuple[str, int, str]) -> str:
    """Classify a problem as easy or hard based on the active SPLIT_METRIC.
      SIZE   → size < 10 → easy  (Paper Ch. 6.1)
      ORACLE → oracle ≤ 5000 → easy
    """
    if SPLIT_METRIC == SplitMetric.SIZE:
        return "easy" if triplet[1] < SMALL_LARGE_THRESHOLD else "hard"
    else:  # ORACLE
        oracle = find_optimal_shots(*triplet)
        return "easy" if oracle <= ORACLE_EASY_THRESHOLD else "hard"

_paper_triplets_cache: Optional[List[Tuple[str, int, str]]] = None

def build_paper_triplets() -> List[Tuple[str, int, str]]:
    """The exact 36 traces from Tables 8-13 of the paper.
    Each algorithm uses specific (size, backend) pairs — NOT a full cartesian product.
    These are used ONLY for evaluation — never seen during training.
    Result is cached after the first call."""
    global _paper_triplets_cache
    if _paper_triplets_cache is not None:
        return _paper_triplets_cache
    _paper_triplets_cache = list(PAPER_TRACES)
    return _paper_triplets_cache

_training_triplets_cache: Optional[List[Tuple[str, int, str]]] = None

def build_training_triplets() -> List[Tuple[str, int, str]]:
    """All qsimbench traces EXCEPT the 36 paper traces.
    Provides 864 diverse problems for training (13 algs, 12 sizes, 6 backends).
    Result is cached after the first call."""
    global _training_triplets_cache
    if _training_triplets_cache is not None:
        return _training_triplets_cache
    index = get_cached_index()
    paper_set = set(build_paper_triplets())
    triplets = []
    for alg in sorted(index.keys()):
        for sz in sorted(index[alg].keys()):
            val = index[alg][sz]
            backends = sorted(val.keys()) if isinstance(val, dict) else sorted(val)
            for be in backends:
                t = (alg, sz, be)
                if t not in paper_set:
                    triplets.append(t)
    _training_triplets_cache = triplets
    return triplets

_validation_set_cache: Dict[str, List[Tuple[str, int, str]]] = {}

def build_validation_set(
    agent_type: AgentType = AgentType.GENERIC,
    seed: int = 42,
) -> List[Tuple[str, int, str]]:
    """Build a validation set from the 36 paper problems for mid-training
    checkpointing.

    The easy/hard selection mirrors the AgentType training distribution so
    that the validation MAE tracks the kind of problems the agent trains on:
      EASY       → all 18 easy paper problems
      HARD       → all 18 hard paper problems
      GENERIC    → all 18 easy + all 18 hard  (full 36)
      UNBALANCED → ~7 easy (20%) + ~29 hard (80%)  sampled from 18+18

    These are the PAPER test traces (never seen during training), so the
    validation signal is a true out-of-distribution check.

    Cached by agent_type.value."""
    cache_key = agent_type.value
    if cache_key in _validation_set_cache:
        return _validation_set_cache[cache_key]

    paper = build_paper_triplets()
    easy = [t for t in paper if classify_problem(t) == "easy"]
    hard = [t for t in paper if classify_problem(t) == "hard"]

    if agent_type == AgentType.EASY:
        result = easy                          # all 18 easy
    elif agent_type == AgentType.HARD:
        result = hard                          # all 18 hard
    elif agent_type == AgentType.UNBALANCED:
        rng = random.Random(seed)
        n_total = len(easy) + len(hard)        # 36
        n_easy = max(1, round(n_total * 0.2))  # ~7
        n_hard = n_total - n_easy              # capped to available
        rng.shuffle(easy)
        rng.shuffle(hard)
        result = easy[:min(n_easy, len(easy))] + hard[:min(n_hard, len(hard))]
    else:  # GENERIC → balanced, use ALL
        result = easy + hard                   # full 36

    _validation_set_cache[cache_key] = result
    return result

_eval_set_cache: Dict[Tuple[str, int], List[Tuple[str, int, str]]] = {}

def build_eval_set(
    agent_type: AgentType,
    seed: int = 42,
) -> List[Tuple[str, int, str]]:
    """Build the evaluation set from the 36 paper problems.

    Always uses ALL 18 easy + ALL 18 hard = 36 problems so that every
    paper problem is evaluated regardless of the agent type.

    Cached by (agent_type.value, seed)."""
    cache_key = (agent_type.value, seed)
    if cache_key in _eval_set_cache:
        return _eval_set_cache[cache_key]

    paper = build_paper_triplets()
    easy = [t for t in paper if classify_problem(t) == "easy"]
    hard = [t for t in paper if classify_problem(t) == "hard"]

    # Use ALL easy + ALL hard = full 36 paper problems
    selected = easy + hard
    _eval_set_cache[cache_key] = selected
    return selected


def get_config_label(agent_type: AgentType = None, for_svg: bool = False) -> str:
    """Return a one-line label describing the current run configuration.
    Used in SVG headers, plot titles, and dashboard suptitles.
    If for_svg=True, uses XML-safe characters (&lt; instead of <)."""
    atype = (agent_type or AGENT_TYPE).value.upper()
    reward = REWARD_TYPE.value.upper()
    if SPLIT_METRIC == SplitMetric.SIZE:
        lt = "&lt;" if for_svg else "<"
        split = f"SIZE (size {lt} {SMALL_LARGE_THRESHOLD})"
    else:
        lte = "&lt;=" if for_svg else "≤"
        split = f"ORACLE (oracle {lte} {ORACLE_EASY_THRESHOLD})"
    return f"Agent: {atype}  |  Reward: {reward}  |  Split: {split}"


def format_training_split_stats(
    train_triplets: List[Tuple[str, int, str]],
    indent: str = "   ",
) -> str:
    """Return a formatted string describing the training set composition
    based on the active SPLIT_METRIC.

      ORACLE → single line:  < 5000 shots (oracle) : N/total (N%)  [NE + NH]
      SIZE   → table by circuit size with count and percentage
    """
    n_total = len(train_triplets)

    if SPLIT_METRIC == SplitMetric.ORACLE:
        n_low = sum(1 for t in train_triplets
                    if ORACLE_CACHE.get(t, find_optimal_shots(*t)) < LOW_SHOT_THRESHOLD)
        n_low_easy = sum(1 for t in train_triplets
                         if classify_problem(t) == "easy"
                         and ORACLE_CACHE.get(t, find_optimal_shots(*t)) < LOW_SHOT_THRESHOLD)
        n_low_hard = sum(1 for t in train_triplets
                         if classify_problem(t) == "hard"
                         and ORACLE_CACHE.get(t, find_optimal_shots(*t)) < LOW_SHOT_THRESHOLD)
        return (f"{indent}< {LOW_SHOT_THRESHOLD} shots (oracle) : {n_low}/{n_total} "
                f"({n_low/n_total*100:.1f}%)  "
                f"[{n_low_easy}E + {n_low_hard}H]")
    else:  # SIZE
        from collections import Counter as Ctr
        size_counts = Ctr(t[1] for t in train_triplets)
        sizes = sorted(size_counts.keys())
        lines = [f"{indent}Training problems by circuit size:"]
        lines.append(f"{indent}  {'Size':>6s}  {'Count':>5s}  {'%':>6s}  {'Class':>5s}")
        lines.append(f"{indent}  {'─'*30}")
        for sz in sizes:
            cnt = size_counts[sz]
            pct = cnt / n_total * 100
            cat = "easy" if sz < SMALL_LARGE_THRESHOLD else "hard"
            lines.append(f"{indent}  q={sz:<4d}  {cnt:>5d}  {pct:>5.1f}%  [{cat}]")
        lines.append(f"{indent}  {'─'*30}")
        n_easy = sum(c for s, c in size_counts.items() if s < SMALL_LARGE_THRESHOLD)
        n_hard = sum(c for s, c in size_counts.items() if s >= SMALL_LARGE_THRESHOLD)
        lines.append(f"{indent}  Total : {n_total}  "
                     f"[{n_easy}E + {n_hard}H]")
        return "\n".join(lines)


def format_training_split_stats_compact(
    train_triplets: List[Tuple[str, int, str]],
) -> str:
    """Compact version for the dashboard text box (monospace, no indent)."""
    n_total = len(train_triplets)

    if SPLIT_METRIC == SplitMetric.ORACLE:
        n_low = sum(1 for t in train_triplets
                    if ORACLE_CACHE.get(t, find_optimal_shots(*t)) < LOW_SHOT_THRESHOLD)
        n_low_easy = sum(1 for t in train_triplets
                         if classify_problem(t) == "easy"
                         and ORACLE_CACHE.get(t, find_optimal_shots(*t)) < LOW_SHOT_THRESHOLD)
        n_low_hard = sum(1 for t in train_triplets
                         if classify_problem(t) == "hard"
                         and ORACLE_CACHE.get(t, find_optimal_shots(*t)) < LOW_SHOT_THRESHOLD)
        return (f"Train < {LOW_SHOT_THRESHOLD} (oracle) : {n_low:>5d}/{n_total}  ({n_low/n_total*100:.1f}%)"
                f"  [{n_low_easy}E+{n_low_hard}H]")
    else:  # SIZE
        from collections import Counter as Ctr
        size_counts = Ctr(t[1] for t in train_triplets)
        sizes = sorted(size_counts.keys())
        lines = ["Train by size:"]
        for sz in sizes:
            cnt = size_counts[sz]
            pct = cnt / n_total * 100
            cat = "E" if sz < SMALL_LARGE_THRESHOLD else "H"
            lines.append(f"  q={sz:<3d}: {cnt:>3d} ({pct:>4.1f}%) [{cat}]")
        n_easy = sum(c for s, c in size_counts.items() if s < SMALL_LARGE_THRESHOLD)
        n_hard = sum(c for s, c in size_counts.items() if s >= SMALL_LARGE_THRESHOLD)
        lines.append(f"  Total: {n_total} [{n_easy}E+{n_hard}H]")
        return "\n".join(lines)

# ─────────────────────────────────────────────────────────────────────────────
# SVG problem overview
# ─────────────────────────────────────────────────────────────────────────────

def generate_problem_svg(
    triplets: List[Tuple[str, int, str]],
    output_path: str = "problem_overview.svg",
) -> str:
    """Create an SVG listing every problem with its easy/hard tag and Oracle value."""
    for t in triplets:
        find_optimal_shots(*t)

    easy = sorted([t for t in triplets if classify_problem(t) == "easy"],
                  key=lambda x: (x[0], x[1], x[2]))
    hard = sorted([t for t in triplets if classify_problem(t) == "hard"],
                  key=lambda x: (x[0], x[1], x[2]))

    row_h, col_w, header_h, padding = 22, 420, 80, 16
    n_rows = max(len(easy), len(hard))
    svg_w  = 2 * col_w + 3 * padding
    svg_h  = header_h + n_rows * row_h + 2 * padding

    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{svg_w}" height="{svg_h}" '
        f'font-family="monospace" font-size="13">',
        f'<rect width="{svg_w}" height="{svg_h}" fill="#fafafa"/>',
        f'<text x="{svg_w//2}" y="20" font-size="14" fill="#37474f" text-anchor="middle">'
        f'{get_config_label(for_svg=True)}</text>',
        f'<line x1="{padding}" y1="28" x2="{svg_w - padding}" y2="28" stroke="#ccc"/>',
    ]

    if SPLIT_METRIC == SplitMetric.SIZE:
        easy_label = f'EASY (size &lt; {SMALL_LARGE_THRESHOLD})'
        hard_label = f'HARD (size &gt;= {SMALL_LARGE_THRESHOLD})'
    else:
        easy_label = f'EASY (oracle &lt;= {ORACLE_EASY_THRESHOLD})'
        hard_label = f'HARD (oracle &gt; {ORACLE_EASY_THRESHOLD})'

    lines += [
        f'<text x="{padding}" y="50" font-size="16" font-weight="bold" fill="#2e7d32">'
        f'{easy_label}  [{len(easy)} problems]</text>',
        f'<text x="{col_w + 2*padding}" y="50" font-size="16" font-weight="bold" fill="#c62828">'
        f'{hard_label}  [{len(hard)} problems]</text>',
        f'<line x1="0" y1="60" x2="{svg_w}" y2="60" stroke="#bbb"/>',
    ]

    def _row(t, idx, x_off):
        alg, sz, be = t
        oracle = ORACLE_CACHE.get((alg, sz, be), "?")
        y = header_h + idx * row_h
        cat = classify_problem(t)
        clr = "#2e7d32" if cat == "easy" else "#c62828"
        tag = "EASY" if cat == "easy" else "HARD"
        return (f'<text x="{x_off}" y="{y}" fill="#333">'
                f'{alg:>8s}  q={sz:<3d}  {be:<18s}  oracle={str(oracle):>6s}'
                f'  <tspan fill="{clr}" font-weight="bold">[{tag}]</tspan></text>')

    for i, t in enumerate(easy):
        lines.append(_row(t, i, padding))
    for i, t in enumerate(hard):
        lines.append(_row(t, i, col_w + 2 * padding))

    lines.append("</svg>")
    with open(output_path, "w") as f:
        f.write("\n".join(lines))
    print(f"SVG problem overview saved -> {output_path}")
    return output_path

# ─────────────────────────────────────────────────────────────────────────────
# Quantum RL Environment  — with incremental cumulative caching
# ─────────────────────────────────────────────────────────────────────────────

class IterativeQuantumEnv(gym.Env):
    """Gym environment for iterative shot allocation.
    Maintains an incremental cumulative counter and caches entropy/variance
    to avoid recomputing from scratch every step."""

    def __init__(
        self,
        triplets: List[Tuple[str, int, str]],
        max_shots: int = MAX_SHOTS,
        step_size: int = 50,
        agent_type: AgentType = AgentType.GENERIC,
        reward_type: RewardType = RewardType.MAE,
        label: str = "env",
    ):
        super().__init__()
        self.max_shots = max_shots
        self.step_size = step_size
        self.agent_type = agent_type
        self.reward_type = reward_type
        self.label = label

        self.active_triplets = triplets
        self.alg_map, self.backend_map = get_global_mappings()

        for t in self.active_triplets:
            find_optimal_shots(*t)

        self.easy_triplets = [t for t in self.active_triplets if classify_problem(t) == "easy"]
        self.hard_triplets = [t for t in self.active_triplets if classify_problem(t) == "hard"]


        # ── Oracle-stratified bins for ORACLE split ─────────────────────
        # When SPLIT_METRIC=ORACLE, the HARD pool is dominated by near-max
        # oracle values (mean~14k, 32% >= 18k).  Stratified sampling across
        # oracle-range bins ensures the agent sees a balanced distribution
        # of stopping points and learns to stop at different budgets.
        self._oracle_bins: Dict[str, Dict[int, List[Tuple[str, int, str]]]] = {}
        if SPLIT_METRIC == SplitMetric.ORACLE and label == "train":
            bin_edges = [0, 5000, 8000, 11000, 14000, 17000, MAX_SHOTS + 1]
            for pool_name, pool in [("easy", self.easy_triplets), ("hard", self.hard_triplets)]:
                bins: Dict[int, List[Tuple[str, int, str]]] = {}
                for t in pool:
                    oracle_val = find_optimal_shots(*t)
                    for j in range(len(bin_edges) - 1):
                        if bin_edges[j] <= oracle_val < bin_edges[j + 1]:
                            bins.setdefault(j, []).append(t)
                            break
                # Store only non-empty bins
                self._oracle_bins[pool_name] = {k: v for k, v in bins.items() if v}

        # Show effective training pool based on agent_type for "train" envs
        if label == "train":
            if agent_type == AgentType.EASY:
                eff = len(self.easy_triplets)
                desc = f"trains on {eff} EASY of {len(self.active_triplets)} pool"
            elif agent_type == AgentType.HARD:
                eff = len(self.hard_triplets)
                desc = f"trains on {eff} HARD of {len(self.active_triplets)} pool"
            elif agent_type == AgentType.UNBALANCED:
                desc = (f"trains 20/80 mix from {len(self.easy_triplets)}E + "
                        f"{len(self.hard_triplets)}H pool")
            else:
                desc = (f"trains 50/50 mix from {len(self.easy_triplets)}E + "
                        f"{len(self.hard_triplets)}H pool")
            # Show oracle-stratified bin distribution when using ORACLE split
            if SPLIT_METRIC == SplitMetric.ORACLE and self._oracle_bins:
                for pk, pk_bins in self._oracle_bins.items():
                    bin_str = ", ".join(f"bin{k}:{len(v)}" for k, v in sorted(pk_bins.items()))
                    desc += f"\n         Oracle bins ({pk}): {bin_str}"
            print(f"[{label.upper()}] {desc}")
        else:
            print(f"[{label.upper()}] {len(self.active_triplets)} problems "
                  f"(easy={len(self.easy_triplets)}, hard={len(self.hard_triplets)})")

        self.action_space = spaces.Discrete(2)
        self.observation_space = spaces.Box(low=0, high=1, shape=(8,), dtype=np.float32)
        self.outcome_history: List[Dict] = []
        self.current_triplet = self.active_triplets[0]
        self.optimal_shots = 0
        self.current_shots = 0

        # ── Incremental caches ──────────────────────────────────────────
        self._cumulative: Counter = Counter()        # running total of all outcomes
        self._cached_entropy: float = 0.5            # last computed entropy
        self._cached_variance: float = 0.5           # last computed variance
        self._cache_valid: bool = False               # True when entropy/variance are up-to-date

    def _stratified_sample(self, pool_name: str) -> Tuple[str, int, str]:
        """Pick a problem from the named pool using oracle-stratified sampling.
        First pick a random bin (uniform), then a random problem within it.
        Falls back to plain random.choice if no bins are available."""
        bins = self._oracle_bins.get(pool_name, {})
        if bins:
            bin_key = random.choice(list(bins.keys()))
            return random.choice(bins[bin_key])
        pool = self.hard_triplets if pool_name == "hard" else self.easy_triplets
        return random.choice(pool or self.active_triplets)

    def reset(self) -> np.ndarray:
        if self.label == "train":
            use_stratified = (SPLIT_METRIC == SplitMetric.ORACLE and self._oracle_bins)

            if self.agent_type == AgentType.EASY:
                if self.easy_triplets:
                    self.current_triplet = (self._stratified_sample("easy")
                                            if use_stratified else
                                            random.choice(self.easy_triplets))
                else:
                    self.current_triplet = random.choice(self.active_triplets)
            elif self.agent_type == AgentType.HARD:
                if self.hard_triplets:
                    self.current_triplet = (self._stratified_sample("hard")
                                            if use_stratified else
                                            random.choice(self.hard_triplets))
                else:
                    self.current_triplet = random.choice(self.active_triplets)
            elif self.agent_type == AgentType.UNBALANCED:
                if random.random() < 0.2 and self.easy_triplets:
                    self.current_triplet = (self._stratified_sample("easy")
                                            if use_stratified else
                                            random.choice(self.easy_triplets))
                elif self.hard_triplets:
                    self.current_triplet = (self._stratified_sample("hard")
                                            if use_stratified else
                                            random.choice(self.hard_triplets))
                else:
                    self.current_triplet = random.choice(self.active_triplets)
            else:  # GENERIC — balanced 50/50
                if random.random() < 0.5 and self.easy_triplets:
                    self.current_triplet = (self._stratified_sample("easy")
                                            if use_stratified else
                                            random.choice(self.easy_triplets))
                elif self.hard_triplets:
                    self.current_triplet = (self._stratified_sample("hard")
                                            if use_stratified else
                                            random.choice(self.hard_triplets))
                else:
                    self.current_triplet = random.choice(self.active_triplets)
        else:
            self.current_triplet = random.choice(self.active_triplets)

        self.optimal_shots = find_optimal_shots(*self.current_triplet)
        self.current_shots = 0
        self.outcome_history = []
        # Reset incremental caches
        self._cumulative = Counter()
        self._cached_entropy = 0.5
        self._cached_variance = 0.5
        self._cache_valid = False
        return self._get_state()

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        if action == 1:  # STOP
            return self._terminate()

        self.current_shots += self.step_size
        try:
            alg, size, backend = self.current_triplet
            batch = get_outcomes(alg, size, backend,
                                 shots=self.step_size, strategy="random", exact=True)
            self.outcome_history.append(batch)
            # Incrementally update cumulative counter — no need to rebuild
            self._cumulative.update(batch)
            self._cache_valid = False  # invalidate entropy/variance cache
        except Exception:
            pass

        if self.current_shots >= self.max_shots:
            return self._terminate()

        return self._get_state(), 0.0, False, {}

    def _terminate(self) -> Tuple[np.ndarray, float, bool, Dict]:
        error = self.current_shots - self.optimal_shots
        mae = abs(error)

        if self.reward_type == RewardType.ASYMMETRIC:
            if error < 0:
                scale = 1.0 + 0.5 * self.optimal_shots / self.max_shots
                final_reward = (-1.85 - (mae / max(self.optimal_shots, 1))) * scale
            else:
                final_reward = np.exp(-0.00025 * error)
        else:
            if error < 0:
                final_reward = -1.0 - (mae / max(self.optimal_shots, 1))
            else:
                final_reward = np.exp(-0.003 * mae)

        info = {
            "shots_used": self.current_shots,
            "optimal_shots": self.optimal_shots,
            "error": error,
            "mae": mae,
            "error_pct": mae / self.max_shots * 100,
            "final_reward": final_reward,
            "triplet": self.current_triplet,
            "difficulty": classify_problem(self.current_triplet),
        }
        return self._get_state(), final_reward, True, info

    # ── state features (with caching) ───────────────────────────────────

    def _compute_distribution_entropy(self, outcomes: Dict[str, int]) -> float:
        total = sum(outcomes.values())
        if total == 0:
            return 0.0
        entropy = 0.0
        for count in outcomes.values():
            if count > 0:
                p = count / total
                entropy -= p * np.log2(p)
        return min(entropy / 4.0, 1.0)

    def _compute_distribution_variance(self, outcomes: Dict[str, int]) -> float:
        if not outcomes:
            return 0.0
        total = sum(outcomes.values())
        if total == 0:
            return 0.0
        probs = [count / total for count in outcomes.values()]
        return min(float(np.var(probs)) * 10.0, 1.0)

    def _update_cached_features(self):
        """Recompute entropy & variance from the incremental cumulative counter
        only when the cache has been invalidated."""
        if self._cache_valid:
            return
        if self._cumulative:
            self._cached_entropy = self._compute_distribution_entropy(self._cumulative)
            self._cached_variance = self._compute_distribution_variance(self._cumulative)
        else:
            self._cached_entropy = 0.5
            self._cached_variance = 0.5
        self._cache_valid = True

    def _compute_rate_of_change(self) -> float:
        if len(self.outcome_history) < 2:
            return 1.0
        # Use cached cumulative minus the last batch for the "previous" distribution
        # instead of rebuilding from scratch
        recent = self.outcome_history[-1]
        recent_total = sum(recent.values())
        if recent_total == 0:
            return 1.0

        prev_cumulative = self._cumulative - Counter(recent)
        prev_total = sum(prev_cumulative.values())
        if prev_total == 0:
            return 1.0

        all_keys = set(prev_cumulative.keys()) | set(recent.keys())
        tvd = 0.0
        for k in all_keys:
            tvd += abs(prev_cumulative.get(k, 0) / prev_total - recent.get(k, 0) / recent_total)
        return 0.5 * tvd

    def _get_state(self) -> np.ndarray:
        alg, size, backend = self.current_triplet
        n_algs  = max(len(self.alg_map), 2)
        n_backs = max(len(self.backend_map), 2)
        alg_norm     = self.alg_map.get(alg, 0) / (n_algs - 1)
        size_norm    = size / 15.0
        backend_norm = self.backend_map.get(backend, 0) / (n_backs - 1)
        shots_norm   = self.current_shots / self.max_shots

        # Use cached entropy/variance (recomputed only when invalidated)
        self._update_cached_features()
        entropy  = self._cached_entropy
        variance = self._cached_variance

        rate     = self._compute_rate_of_change()
        progress = min(self.current_shots / (size * 500), 1.0)

        return np.array(
            [alg_norm, size_norm, backend_norm, shots_norm,
             entropy, variance, rate, progress],
            dtype=np.float32,
        )

# ─────────────────────────────────────────────────────────────────────────────
# DQN Network & Agent
# ─────────────────────────────────────────────────────────────────────────────

class DQN(nn.Module):
    def __init__(self, input_size: int, output_size: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 256),        nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128),        nn.ReLU(),
            nn.Linear(128, output_size),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Agent:
    def __init__(
        self,
        state_size: int = 8,
        action_size: int = 2,
        learning_rate: float = 1e-4,
        gamma: float = 0.99,
    ):
        self.q_net = DQN(state_size, action_size)
        self.target_net = DQN(state_size, action_size)
        self.update_target()
        self.memory: deque = deque(maxlen=200_000)
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=learning_rate)
        self.gamma = gamma
        self.epsilon = 1.0
        self.epsilon_decay = 0.9998
        self.epsilon_min = 0.05
        self.action_size = action_size

    def act(self, state: np.ndarray, evaluate: bool = False) -> int:
        if not evaluate and random.random() < self.epsilon:
            return random.randint(0, self.action_size - 1)
        state_t = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q = self.q_net(state_t)
        return int(torch.argmax(q).item())

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def replay(self, batch_size: int = 64):
        if len(self.memory) < batch_size:
            return
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        states      = torch.FloatTensor(np.array(states))
        actions     = torch.LongTensor(actions)
        rewards     = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(np.array(next_states))
        dones       = torch.FloatTensor(dones)

        cur_q = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target_net(next_states).max(1)[0]
            target_q = rewards + (1 - dones) * self.gamma * next_q

        loss = F.mse_loss(cur_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

    def get_weights(self) -> dict:
        return copy.deepcopy(self.q_net.state_dict())

    def load_weights(self, state_dict: dict):
        self.q_net.load_state_dict(state_dict)
        self.update_target()

# ─────────────────────────────────────────────────────────────────────────────
# Snapshot Validator  (uses paper problems filtered by AgentType for checkpointing)
# ─────────────────────────────────────────────────────────────────────────────

class SnapshotValidator:
    """Validates agent on a fixed validation subset; returns MAE.
    Iterates deterministically through every triplet (not random sampling)
    so that the validation MAE is reproducible across episodes."""

    def __init__(self, validation_env: IterativeQuantumEnv):
        self.env = validation_env
        self.history: List[Dict] = []

    def validate(self, agent: Agent, episode: int) -> float:
        total_mae, total_reward = 0.0, 0.0
        n = len(self.env.active_triplets)
        orig_eps = agent.epsilon
        agent.epsilon = 0.0

        for triplet in self.env.active_triplets:
            # Deterministic: set the exact triplet instead of random reset
            self.env.current_triplet = triplet
            self.env.optimal_shots = find_optimal_shots(*triplet)
            self.env.current_shots = 0
            self.env.outcome_history = []
            self.env._cumulative = Counter()
            self.env._cache_valid = False
            state = self.env._get_state()

            done, info = False, {}
            ep_reward = 0.0
            while not done:
                action = agent.act(state, evaluate=True)
                state, reward, done, info = self.env.step(action)
                ep_reward += reward
            total_mae += abs(info.get("error", 0))
            total_reward += ep_reward

        agent.epsilon = orig_eps

        avg_mae    = total_mae / max(n, 1)
        avg_reward = total_reward / max(n, 1)
        error_pct  = avg_mae / self.env.max_shots * 100

        self.history.append({
            "episode": episode, "mae": avg_mae,
            "error_pct": error_pct, "reward": avg_reward,
        })
        return avg_mae

# ─────────────────────────────────────────────────────────────────────────────
# Training loop
# ─────────────────────────────────────────────────────────────────────────────

def train_agent(
    agent_type: AgentType = AgentType.GENERIC,
    reward_type: RewardType = RewardType.MAE,
    num_episodes: int = 3000,
    update_target_every: int = 500,
    snapshot_interval: int = 200,
    checkpoint_start: int = 2000,
    checkpoint_interval: int = 100,
    patience: int = 5,
    seed: int = 42,
) -> Tuple[Agent, IterativeQuantumEnv, List[float], List[Dict]]:
    """
    Train a DQN agent on the 864 non-paper traces from qsimbench.
    agent_type controls training distribution:
      EASY       → 100% easy problems
      HARD       → 100% hard problems
      GENERIC    → 50/50 balanced mix
      UNBALANCED → 20% easy, 80% hard
    Validation for checkpointing uses the 36 PAPER test problems,
    filtered by the same easy/hard ratio as the training distribution
    (e.g. HARD → all 18 hard paper problems).
    """
    load_oracle_cache()

    train_triplets = build_training_triplets()
    val_triplets   = build_validation_set(agent_type=agent_type, seed=seed)

    env     = IterativeQuantumEnv(train_triplets, agent_type=agent_type, reward_type=reward_type, label="train")
    val_env = IterativeQuantumEnv(val_triplets,   reward_type=reward_type, label="validation")

    agent     = Agent()
    validator = SnapshotValidator(val_env)

    rewards_history: List[float] = []
    best_mae: float = float("inf")
    best_weights: Optional[dict] = None
    no_improve_count: int = 0
    stopped_early = False

    # ── Describe effective training distribution ───────────────────────
    n_easy_pool = len(env.easy_triplets)
    n_hard_pool = len(env.hard_triplets)
    if agent_type == AgentType.EASY:
        train_desc = f"{n_easy_pool} EASY problems (100% easy)"
    elif agent_type == AgentType.HARD:
        train_desc = f"{n_hard_pool} HARD problems (100% hard)"
    elif agent_type == AgentType.UNBALANCED:
        train_desc = f"20% easy / 80% hard from {n_easy_pool}E + {n_hard_pool}H pool"
    else:
        train_desc = f"50/50 balanced from {n_easy_pool}E + {n_hard_pool}H pool"

    print(f"\n{'='*60}")
    print(f" Agent Type  : {agent_type.value.upper()}")
    print(f" Reward Type : {reward_type.value.upper()}")
    print(f" Split Metric: {SPLIT_METRIC.value.upper()}")
    print(f" Train       : {train_desc}")
    print(f" Validation  : {len(val_triplets)} paper problems (filtered by agent type)")
    print(f" Test        : {len(build_paper_triplets())} paper problems (ALL, evaluated after training)")
    print(f" Episodes    : {num_episodes}  |  Patience : {patience}")
    print(f"{'='*60}\n")

    for episode in tqdm(range(num_episodes), desc="Training"):
        state = env.reset()
        total_reward = 0.0
        done = False

        while not done:
            action = agent.act(state)
            next_state, reward, done, info = env.step(action)
            agent.remember(state, action, reward, next_state, done)
            agent.replay(batch_size=64)
            state = next_state
            total_reward += reward

        rewards_history.append(total_reward)

        if episode > 0 and episode % update_target_every == 0:
            agent.update_target()

        # ── Periodic snapshot validation ────────────────────────────────
        # Skip snapshot if checkpoint will also fire this episode (avoids
        # duplicate validator.validate() calls and double history entries)
        checkpoint_fires = (episode >= checkpoint_start
                            and episode % checkpoint_interval == 0)
        if episode % snapshot_interval == 0 and not checkpoint_fires:
            current_mae = validator.validate(agent, episode)
            error_pct = current_mae / env.max_shots * 100

            if len(validator.history) > 1:
                prev = validator.history[-2]
                delta = current_mae - prev["mae"]
                icon = "↓" if delta < 0 else "↑"
                print(f"\n[Snapshot Ep {episode}] Val MAE: {current_mae:.1f} "
                      f"({error_pct:.2f}%)  Δ: {delta:+.1f}  {icon}")
            else:
                print(f"\n[Snapshot Ep {episode}] Val MAE: {current_mae:.1f} "
                      f"({error_pct:.2f}%)  (Baseline)")

        # ── Best-model checkpointing ────────────────────────────────────
        if episode >= checkpoint_start and episode % checkpoint_interval == 0:
            ckpt_mae = validator.validate(agent, episode)

            if ckpt_mae < best_mae:
                best_mae = ckpt_mae
                best_weights = agent.get_weights()
                no_improve_count = 0
                print(f"  ★ New best @ ep {episode}: MAE={best_mae:.1f} "
                      f"({best_mae/env.max_shots*100:.2f}%)")
            else:
                no_improve_count += 1
                print(f"  ✗ No improvement ({no_improve_count}/{patience}) "
                      f"current={ckpt_mae:.1f} vs best={best_mae:.1f}")

            if no_improve_count >= patience:
                print(f"\n>> Early stopping at episode {episode} (patience={patience})")
                stopped_early = True
                break

    save_oracle_cache()

    if best_weights is not None:
        agent.load_weights(best_weights)
        print(f"\n>> Restored best model weights (MAE={best_mae:.1f})")
    else:
        print("\n>> No checkpoint taken (training ended before checkpoint_start)")

    print(f"Training {'ended early at ep ' + str(episode) if stopped_early else 'completed all ' + str(num_episodes) + ' episodes'}.")

    return agent, env, rewards_history, validator.history

# ─────────────────────────────────────────────────────────────────────────────
# Evaluation on the 36 paper problems
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_agent(agent: Agent, agent_type: AgentType = AgentType.GENERIC, seed: int = 42) -> List[Dict]:
    """Evaluate the trained agent on ALL 36 paper problems (18 easy + 18 hard).
    The full test set is always used regardless of agent type so that results
    are directly comparable across training configurations.
    The agent has never seen these traces during training."""
    eval_triplets = build_eval_set(agent_type, seed=seed)
    n_easy = sum(1 for t in eval_triplets if classify_problem(t) == "easy")
    n_hard = sum(1 for t in eval_triplets if classify_problem(t) == "hard")

    env_eval = IterativeQuantumEnv(eval_triplets, label=f"eval ({agent_type.value})")

    results: List[Dict] = []
    for triplet in tqdm(env_eval.active_triplets, desc=f"Evaluating ({agent_type.value}, {len(eval_triplets)} problems)"):
        env_eval.current_triplet = triplet
        env_eval.optimal_shots = find_optimal_shots(*triplet)
        env_eval.current_shots = 0
        env_eval.outcome_history = []
        env_eval._cumulative = Counter()
        env_eval._cache_valid = False
        state = env_eval._get_state()

        done, info = False, {}
        while not done:
            action = agent.act(state, evaluate=True)
            state, _, done, info = env_eval.step(action)
        results.append(info)

    # Summary
    maes = [r["mae"] for r in results]
    easy_maes = [r["mae"] for r in results if r["difficulty"] == "easy"]
    hard_maes = [r["mae"] for r in results if r["difficulty"] == "hard"]
    avg_mae = np.mean(maes)
    pct = avg_mae / env_eval.max_shots * 100

    train_triplets = build_training_triplets()

    print(f"\n{'='*55}")
    print(f"  TEST EVALUATION — {agent_type.value.upper()} agent")
    print(f"  {len(eval_triplets)} problems  ({n_easy} easy + {n_hard} hard)")
    print(f"  {'─'*50}")
    print(f"  Overall MAE   : {avg_mae:.1f} shots  ({pct:.2f}%)")
    if easy_maes:
        print(f"  Easy MAE      : {np.mean(easy_maes):.1f} shots  ({len(easy_maes)} problems)")
    if hard_maes:
        print(f"  Hard MAE      : {np.mean(hard_maes):.1f} shots  ({len(hard_maes)} problems)")
    print(f"  {'─'*50}")
    print(format_training_split_stats(train_triplets, indent="  "))
    print(f"{'='*55}")
    return results


def evaluate_on_triplets(agent: Agent, triplets: List[Tuple[str, int, str]],
                         desc: str = "Evaluating") -> List[Dict]:
    """Run the trained agent greedily on *triplets* and return per-problem info dicts.
    This is a lightweight evaluation loop (no printed summary) suitable for
    getting scatter-plot data on the training set or any arbitrary set."""
    env_tmp = IterativeQuantumEnv(triplets, label=desc)
    results: List[Dict] = []
    for triplet in tqdm(env_tmp.active_triplets, desc=desc):
        env_tmp.current_triplet = triplet
        env_tmp.optimal_shots = find_optimal_shots(*triplet)
        env_tmp.current_shots = 0
        env_tmp.outcome_history = []
        env_tmp._cumulative = Counter()
        env_tmp._cache_valid = False
        state = env_tmp._get_state()
        done, info = False, {}
        while not done:
            action = agent.act(state, evaluate=True)
            state, _, done, info = env_tmp.step(action)
        results.append(info)
    return results

# ─────────────────────────────────────────────────────────────────────────────
# Agent vs Oracle comparison SVG (all 36 paper problems)
# ─────────────────────────────────────────────────────────────────────────────

def generate_comparison_svg(
    agent: Agent,
    triplets: List[Tuple[str, int, str]],
    output_path: str = "agent_vs_oracle.svg",
) -> str:
    """
    Run the trained agent on EVERY problem in `triplets` and produce an SVG
    table comparing agent shots vs oracle shots for each trace.
    """
    env = IterativeQuantumEnv(triplets, label="comparison")
    comparison: List[Dict] = []
    orig_eps = agent.epsilon
    agent.epsilon = 0.0

    for triplet in tqdm(triplets, desc="Agent vs Oracle comparison"):
        env.current_triplet = triplet
        env.optimal_shots = find_optimal_shots(*triplet)
        env.current_shots = 0
        env.outcome_history = []
        env._cumulative = Counter()
        env._cache_valid = False
        state = env._get_state()
        done, info = False, {}
        while not done:
            action = agent.act(state, evaluate=True)
            state, _, done, info = env.step(action)
        comparison.append(info)

    agent.epsilon = orig_eps

    result_map: Dict[Tuple, Dict] = {tuple(r["triplet"]): r for r in comparison}

    easy_trips = sorted([t for t in triplets if classify_problem(t) == "easy"],
                        key=lambda x: (x[0], x[1], x[2]))
    hard_trips = sorted([t for t in triplets if classify_problem(t) == "hard"],
                        key=lambda x: (x[0], x[1], x[2]))

    row_h, col_w, header_h, padding = 20, 540, 100, 16
    n_rows = max(len(easy_trips), len(hard_trips))
    svg_w  = 2 * col_w + 3 * padding
    svg_h  = header_h + n_rows * row_h + 2 * padding

    all_maes  = [r["mae"] for r in comparison]
    easy_maes = [result_map[t]["mae"] for t in easy_trips]
    hard_maes = [result_map[t]["mae"] for t in hard_trips]
    avg_mae  = np.mean(all_maes)
    avg_easy = np.mean(easy_maes) if easy_maes else 0
    avg_hard = np.mean(hard_maes) if hard_maes else 0

    config_lbl = get_config_label(for_svg=True)

    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{svg_w}" height="{svg_h}" '
        f'font-family="monospace" font-size="12">',
        f'<rect width="{svg_w}" height="{svg_h}" fill="#fafafa"/>',
        f'<text x="{svg_w//2}" y="18" font-size="12" fill="#37474f" '
        f'text-anchor="middle">{config_lbl}</text>',
        f'<text x="{svg_w//2}" y="38" font-size="16" font-weight="bold" fill="#1a237e" '
        f'text-anchor="middle">Agent vs Oracle — {len(triplets)} Problems  '
        f'(MAE: {avg_mae:.0f} shots, {avg_mae/MAX_SHOTS*100:.2f}%)</text>',
        f'<text x="{padding}" y="62" font-size="14" font-weight="bold" fill="#2e7d32">'
        f'EASY [{len(easy_trips)}]  avg MAE={avg_easy:.0f}</text>',
        f'<text x="{col_w + 2*padding}" y="62" font-size="14" font-weight="bold" fill="#c62828">'
        f'HARD [{len(hard_trips)}]  avg MAE={avg_hard:.0f}</text>',
        f'<text x="{padding}" y="79" font-size="11" fill="#666">'
        f'{"Algorithm":>8s}  {"q":>3s}  {"Backend":<18s}  {"Oracle":>6s}  {"Agent":>6s}  {"Δ":>7s}</text>',
        f'<text x="{col_w + 2*padding}" y="79" font-size="11" fill="#666">'
        f'{"Algorithm":>8s}  {"q":>3s}  {"Backend":<18s}  {"Oracle":>6s}  {"Agent":>6s}  {"Δ":>7s}</text>',
        f'<line x1="0" y1="85" x2="{svg_w}" y2="85" stroke="#bbb"/>',
    ]

    def _row(triplet, idx, x_off):
        alg, sz, be = triplet
        r = result_map.get(tuple(triplet), {})
        oracle = r.get("optimal_shots", ORACLE_CACHE.get((alg, sz, be), 0))
        agent_shots = r.get("shots_used", 0)
        delta = agent_shots - oracle
        y = header_h + idx * row_h
        ratio = abs(delta) / oracle if oracle > 0 else 0.0
        if ratio <= 0.10:
            clr = "#2e7d32"
        elif ratio <= 0.25:
            clr = "#e65100"
        else:
            clr = "#c62828"
        return (f'<text x="{x_off}" y="{y}" fill="#333">'
                f'{alg:>8s}  {sz:>3d}  {be:<18s}  {oracle:>6d}  {agent_shots:>6d}'
                f'  <tspan fill="{clr}" font-weight="bold">{delta:+7d}</tspan></text>')

    for i, t in enumerate(easy_trips):
        lines.append(_row(t, i, padding))
    for i, t in enumerate(hard_trips):
        lines.append(_row(t, i, col_w + 2 * padding))

    lines.append("</svg>")
    with open(output_path, "w") as f:
        f.write("\n".join(lines))
    print(f"Agent vs Oracle comparison SVG saved -> {output_path}")
    return output_path

# ─────────────────────────────────────────────────────────────────────────────
# Analysis Dashboard
# ─────────────────────────────────────────────────────────────────────────────

class AnalysisDashboard:
    def __init__(self, training_rewards, evaluation_results, validation_history,
                 agent_type: AgentType = AgentType.GENERIC, max_shots=MAX_SHOTS,
                 train_results: Optional[List[Dict]] = None):
        self.rewards     = training_rewards
        self.results     = evaluation_results
        self.val_history = validation_history
        self.agent_type  = agent_type
        self.max_shots   = max_shots
        self.train_results = train_results or []

        self.shots_used    = [r["shots_used"] for r in self.results]
        self.optimal_shots = [r["optimal_shots"] for r in self.results]
        self.errors        = np.array([r["error"] for r in self.results])
        self.maes          = np.abs(self.errors)
        self.difficulties  = [r["difficulty"] for r in self.results]

    def plot_dashboard(self, save_path: str = "dashboard.svg"):
        fig, axes = plt.subplots(2, 4, figsize=(32, 14))
        config_lbl = get_config_label(self.agent_type)
        n_eval = len(self.results)
        fig.suptitle(f"DRL for Shot Allocation — {config_lbl}\n"
                     f"Eval: {n_eval} paper problems  |  Train: 864 non-paper traces",
                     fontsize=18, y=0.99)

        self._plot_training_rewards(axes[0, 0])
        self._plot_snapshot_evolution(axes[0, 1])
        self._plot_performance_scatter(axes[0, 2])
        self._plot_training_scatter(axes[0, 3])
        self._plot_error_distribution(axes[1, 0])
        self._plot_efficiency_curve(axes[1, 1])
        self._plot_mae_summary(axes[1, 2])
        # Bottom-right: leave blank or add a second summary
        axes[1, 3].axis("off")

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        fig.savefig(save_path, format="svg", bbox_inches="tight")
        print(f"Dashboard saved -> {save_path}")
        plt.close(fig)

    def _plot_training_rewards(self, ax):
        ax.plot(self.rewards, alpha=0.3, color="gray", label="Raw")
        if len(self.rewards) >= 100:
            ma = np.convolve(self.rewards, np.ones(100) / 100, mode="valid")
            ax.plot(ma, color="blue", label="Moving Avg (100)")
        ax.set_title("Training Rewards", fontsize=14)
        ax.legend()
        ax.grid(True, alpha=0.3)

    def _plot_snapshot_evolution(self, ax):
        eps  = [x["episode"] for x in self.val_history]
        maes = [x["mae"] for x in self.val_history]
        pcts = [x.get("error_pct", m / self.max_shots * 100)
                for x, m in zip(self.val_history, maes)]
        ax.plot(eps, maes, marker="o", color="green", linewidth=2)
        ax.set_title("Snapshot Validation: Model Evolution", fontsize=14)
        ax.set_ylabel("MAE on Validation Set (shots)")
        ax.set_xlabel("Training Episode")
        ax.grid(True, alpha=0.5)

        ax2 = ax.twinx()
        ax2.plot(eps, pcts, marker="x", color="orange", linewidth=1,
                 alpha=0.6, label="Error %")
        ax2.set_ylabel("Error %  (MAE / max_shots × 100)")
        ax2.legend(loc="upper right")

        if len(maes) > 1:
            ax.text(0.05, 0.95,
                    f"Best MAE: {min(maes):.1f}\nImprov: {maes[0]-min(maes):.1f}",
                    transform=ax.transAxes,
                    bbox=dict(facecolor="white", alpha=0.8),
                    verticalalignment="top")

    def _plot_performance_scatter(self, ax):
        """Policy vs Oracle scatter on EVALUATION set — color-coded by easy/hard."""
        easy_idx = [i for i, d in enumerate(self.difficulties) if d == "easy"]
        hard_idx = [i for i, d in enumerate(self.difficulties) if d == "hard"]

        opt = np.array(self.optimal_shots)
        used = np.array(self.shots_used)

        if easy_idx:
            ax.scatter(opt[easy_idx], used[easy_idx], alpha=0.6,
                       edgecolors="k", c="#4caf50", label=f"Easy ({len(easy_idx)})",
                       s=40)
        if hard_idx:
            ax.scatter(opt[hard_idx], used[hard_idx], alpha=0.6,
                       edgecolors="k", c="#e53935", label=f"Hard ({len(hard_idx)})",
                       s=40)

        m = max(max(self.optimal_shots), max(self.shots_used))
        ax.plot([0, m], [0, m], "k--", alpha=0.5, label="Ideal")

        mae_val = float(np.mean(self.maes))
        pct = mae_val / self.max_shots * 100
        n_eval = len(self.results)
        ax.set_title(f"EVAL SET ({n_eval} probs)  —  MAE: {mae_val:.0f}  ({pct:.2f}%)",
                     fontsize=14)
        ax.set_xlabel("Oracle Optimal Shots")
        ax.set_ylabel("Agent Shots")
        ax.legend()
        ax.grid(True, alpha=0.3)

    def _plot_training_scatter(self, ax):
        """Policy vs Oracle scatter on TRAINING set — to check for overfitting.
        If the agent performs much better here than on the eval set, it's overfitting."""
        if not self.train_results:
            ax.text(0.5, 0.5, "No training-set\nevaluation data",
                    transform=ax.transAxes, ha="center", va="center", fontsize=14, color="gray")
            ax.set_title("TRAIN SET — No Data", fontsize=14)
            ax.axis("off")
            return

        tr_shots = [r["shots_used"] for r in self.train_results]
        tr_optimal = [r["optimal_shots"] for r in self.train_results]
        tr_diff = [r["difficulty"] for r in self.train_results]
        tr_errors = np.array([r["error"] for r in self.train_results])
        tr_maes = np.abs(tr_errors)

        easy_idx = [i for i, d in enumerate(tr_diff) if d == "easy"]
        hard_idx = [i for i, d in enumerate(tr_diff) if d == "hard"]
        opt = np.array(tr_optimal)
        used = np.array(tr_shots)

        if easy_idx:
            ax.scatter(opt[easy_idx], used[easy_idx], alpha=0.4,
                       edgecolors="k", c="#4caf50", label=f"Easy ({len(easy_idx)})",
                       s=25)
        if hard_idx:
            ax.scatter(opt[hard_idx], used[hard_idx], alpha=0.4,
                       edgecolors="k", c="#e53935", label=f"Hard ({len(hard_idx)})",
                       s=25)

        m = max(max(tr_optimal), max(tr_shots))
        ax.plot([0, m], [0, m], "k--", alpha=0.5, label="Ideal")

        train_mae = float(np.mean(tr_maes))
        train_pct = train_mae / self.max_shots * 100
        eval_mae = float(np.mean(self.maes))
        eval_pct = eval_mae / self.max_shots * 100
        gap = train_pct - eval_pct

        ax.set_title(f"TRAIN SET ({len(self.train_results)} probs)  —  MAE: {train_mae:.0f}  ({train_pct:.2f}%)",
                     fontsize=14)
        ax.set_xlabel("Oracle Optimal Shots")
        ax.set_ylabel("Agent Shots")

        # Show overfit indicator
        icon = "⚠️ OVERFIT" if gap < -2.0 else "✓ OK"
        ax.text(0.05, 0.95,
                f"Train err: {train_pct:.2f}%\nEval err : {eval_pct:.2f}%\nΔ: {gap:+.2f}%  {icon}",
                transform=ax.transAxes, fontsize=10, verticalalignment="top",
                fontfamily="monospace",
                bbox=dict(facecolor="lightyellow", alpha=0.9, edgecolor="gray"))
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    def _plot_error_distribution(self, ax):
        ax.hist(self.errors, bins=40, color="purple", alpha=0.7, edgecolor="black")
        ax.axvline(0, color="r", linestyle="--")
        ax.set_title("Error Distribution (Residuals)", fontsize=14)
        ax.set_xlabel("Error (Agent − Oracle)")

    def _plot_efficiency_curve(self, ax):
        safe_opt = np.where(np.array(self.optimal_shots) == 0, 1, self.optimal_shots)
        ratios = np.array(self.shots_used) / safe_opt
        ax.hist(ratios, bins=50, color="orange", alpha=0.7, edgecolor="black")
        ax.axvline(1.0, color="r", linewidth=2, linestyle="--", label="Ideal (1.0)")
        ax.set_title("Efficiency Ratio Distribution", fontsize=14)
        ax.set_xlabel("Ratio (Agent / Oracle)")
        ax.legend()
        ax.grid(True, alpha=0.3)

    def _plot_mae_summary(self, ax):
        mae_val    = float(np.mean(self.maes))
        median_mae = float(np.median(self.maes))
        pct        = mae_val / self.max_shots * 100
        pct_med    = median_mae / self.max_shots * 100

        easy_maes = [self.maes[i] for i, d in enumerate(self.difficulties) if d == "easy"]
        hard_maes = [self.maes[i] for i, d in enumerate(self.difficulties) if d == "hard"]

        n_under = int(np.sum(self.errors < 0))
        n_over  = int(np.sum(self.errors > 0))
        n_exact = int(np.sum(self.errors == 0))

        n_easy = len(easy_maes)
        n_hard = len(hard_maes)
        n_total = n_easy + n_hard

        train_triplets = build_training_triplets()
        split_stats = format_training_split_stats_compact(train_triplets)

        txt = (
            f"EVAL: {n_total} problems ({n_easy}E + {n_hard}H)\n"
            f"Agent Type   : {self.agent_type.value.upper()}\n"
            f"Split Metric : {SPLIT_METRIC.value.upper()}\n"
            f"{'─'*36}\n"
            f"Mean MAE     : {mae_val:>8.1f} shots\n"
            f"Median MAE   : {median_mae:>8.1f} shots\n"
            f"Error %      : {pct:>8.2f}%  (mean)\n"
            f"Error % med  : {pct_med:>8.2f}%  (median)\n"
            f"{'─'*36}\n"
        )
        if easy_maes:
            txt += f"Easy MAE     : {np.mean(easy_maes):>8.1f}  ({n_easy} probs)\n"
        if hard_maes:
            txt += f"Hard MAE     : {np.mean(hard_maes):>8.1f}  ({n_hard} probs)\n"
        txt += (
            f"{'─'*36}\n"
            f"Undershoot   : {n_under:>5d}  ({n_under/len(self.errors)*100:.1f}%)\n"
            f"Overshoot    : {n_over:>5d}  ({n_over/len(self.errors)*100:.1f}%)\n"
            f"Exact        : {n_exact:>5d}  ({n_exact/len(self.errors)*100:.1f}%)\n"
            f"{'─'*36}\n"
            f"{split_stats}\n"
            f"{'─'*36}\n"
            f"Trained on   :  864 non-paper traces\n"
            f"max_shots    : {self.max_shots}"
        )
        ax.text(0.05, 0.95, txt, transform=ax.transAxes,
                fontsize=12, fontfamily="monospace", verticalalignment="top",
                bbox=dict(facecolor="lightyellow", alpha=0.9, edgecolor="gray"))
        ax.set_title("MAE & Error %", fontsize=14)
        ax.axis("off")

# ─────────────────────────────────────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── Configuration ───────────────────────────────────────────────────
    NUM_EPISODES        = 3000
    CHECKPOINT_START    = 1200
    CHECKPOINT_INTERVAL = 100
    PATIENCE            = 5
    SEED                = 42
    AGENT_TYPE          = AgentType.HARD   # EASY / HARD / GENERIC / UNBALANCED
    REWARD_TYPE         = RewardType.MAE      # MAE / ASYMMETRIC
    SPLIT_METRIC        = SplitMetric.ORACLE       # SIZE / ORACLE
    # ────────────────────────────────────────────────────────────────────

    # 0. Generate SVG of all 36 paper problems (test set)
    paper_triplets = build_paper_triplets()
    generate_problem_svg(paper_triplets, output_path="problem_overview.svg")

    # 1. Train on 864 non-paper problems (distribution bias set by AGENT_TYPE)
    agent, train_env, rewards, val_history = train_agent(
        agent_type=AGENT_TYPE,
        reward_type=REWARD_TYPE,
        num_episodes=NUM_EPISODES,
        update_target_every=500,  
        snapshot_interval=200,
        checkpoint_start=CHECKPOINT_START,
        checkpoint_interval=CHECKPOINT_INTERVAL,
        patience=PATIENCE,
        seed=SEED,
    )

    # 2. Evaluate on ALL 36 paper problems
    results = evaluate_agent(agent, agent_type=AGENT_TYPE, seed=SEED)

    # 2b. Evaluate on training set (overfitting diagnostic)
    train_triplets = build_training_triplets()
    train_results = evaluate_on_triplets(agent, train_triplets,
                                         desc="Train-set eval (overfitting check)")

    # 3. Dashboard
    tag = AGENT_TYPE.value  # "easy", "hard", or "generic"
    os.makedirs(tag, exist_ok=True)
    dash = AnalysisDashboard(rewards, results, val_history,
                             agent_type=AGENT_TYPE, train_results=train_results)
    dash.plot_dashboard(save_path=f"{tag}/dashboard-{tag}.svg")

    # 4. Detailed per-problem comparison SVG (same eval subset)
    eval_triplets = build_eval_set(AGENT_TYPE, seed=SEED)
    generate_comparison_svg(agent, eval_triplets, output_path=f"{tag}/agent_vs_oracle-{tag}.svg")

SVG problem overview saved -> problem_overview.svg
Loaded 900 items from oracle cache.
[TRAIN] trains on 399 HARD of 864 pool
         Oracle bins (easy): bin0:463, bin1:2
         Oracle bins (hard): bin1:69, bin2:44, bin3:58, bin4:64, bin5:164
[VALIDATION] 20 problems (easy=0, hard=20)

 Agent Type  : HARD
 Reward Type : MAE
 Split Metric: ORACLE
 Train       : 399 HARD problems (100% hard)
 Validation  : 20 paper problems (filtered by agent type)
 Test        : 36 paper problems (ALL, evaluated after training)
 Episodes    : 3000  |  Patience : 5



Training:   1%|          | 35/3000 [00:05<06:01,  8.20it/s] 


[Snapshot Ep 0] Val MAE: 10700.0 (53.50%)  (Baseline)


Training:   7%|▋         | 211/3000 [01:05<47:07,  1.01s/it]  


[Snapshot Ep 200] Val MAE: 6310.0 (31.55%)  Δ: -4390.0  ↓


...


  TEST EVALUATION — HARD agent
  36 problems  (16 easy + 20 hard)
  ──────────────────────────────────────────────────
  Overall MAE   : 11769.4 shots  (58.85%)
  Easy MAE      : 18593.8 shots  (16 problems)
  Hard MAE      : 6310.0 shots  (20 problems)
  ──────────────────────────────────────────────────
  < 5000 shots (oracle) : 463/864 (53.6%)  [463E + 0H]
[TRAIN-SET EVAL (OVERFITTING CHECK)] 864 problems (easy=465, hard=399)


Train-set eval (overfitting check): 100%|██████████| 864/864 [24:15<00:00,  1.68s/it]



Dashboard saved -> hard/dashboard-hard.svg
[COMPARISON] 36 problems (easy=16, hard=20)


Agent vs Oracle comparison: 100%|██████████| 36/36 [01:14<00:00,  2.08s/it]

Agent vs Oracle comparison SVG saved -> hard/agent_vs_oracle-hard.svg
